In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
import ast
import re 
#plt.rcParams['font.family'] = 'AppleGothic'    # Mac
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

reviews = pd.read_csv("../../../data/steam_indie_reviews.csv")

# 1. 파일 불러오기
reviews = pd.read_csv("../../../data/steam_indie_reviews.csv")

In [36]:
# 2. 원본 복사
reviews_clean = reviews.copy()

In [37]:
# 3. 중복 제거
reviews_clean = reviews_clean.drop_duplicates(subset=["recommendationid"])

# 4. 리뷰 결측 제거
reviews_clean = reviews_clean.dropna(subset=["review"])


In [38]:
# 5. 긍정/부정 라벨 생성
reviews_clean["sentiment"] = reviews_clean["voted_up"].map({
    True: "positive",
    False: "negative"
})

In [39]:
# 6. 날짜 변환
date_cols = [
    "timestamp_created",
    "timestamp_updated",
    "author_last_played"
]

for col in date_cols:
    reviews_clean[col.replace("timestamp_", "") + "_date"] = pd.to_datetime(
        reviews_clean[col],
        unit="s",
        errors="coerce"
    )

# author_last_played는 이름 따로 정리
reviews_clean["last_played_date"] = pd.to_datetime(
    reviews_clean["author_last_played"],
    unit="s",
    errors="coerce"
)

In [40]:
# 7. 작성 월 생성
reviews_clean["created_month"] = reviews_clean["created_date"].dt.to_period("M").astype(str)
reviews_clean["created_year"] = reviews_clean["created_date"].dt.year


In [41]:
# 8. 플레이타임 분 단위 → 시간 단위 변환
playtime_cols = [
    "author_playtime_forever",
    "author_playtime_last_two_weeks",
    "author_playtime_at_review"
]

for col in playtime_cols:
    reviews_clean[col.replace("author_playtime_", "playtime_") + "_hours"] = (
        reviews_clean[col] / 60
    )


In [42]:
# 9. 리뷰 길이 생성
reviews_clean["review_length"] = reviews_clean["review"].astype(str).str.len()
reviews_clean["word_count"] = reviews_clean["review"].astype(str).str.split().str.len()


영어인 리뷰만 뽑아보기

In [43]:
reviews_en = reviews_clean[
    (reviews_clean["language"] == "english") &
    (reviews_clean["review"].notna())
].copy()

print(reviews_en.shape)
reviews_en[["review", "sentiment"]].head()

(148915, 33)


,review,sentiment
4,"Fun fact, if you wanted to 100% this game, at ...",negative
24,"i'd give it a 3.5. Not terrible, not great either",negative
27,"Really fun game. Grindier as hell, but fun.",positive
29,Honestly not a bad game just kinda boring ngl,negative
31,Was having fun until I got an obnoxious rocket...,negative


In [44]:
# 10. 텍스트 정리 함수
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

reviews_clean["clean_review"] = reviews_clean["review"].apply(clean_text)


In [45]:
# 12. 너무 짧은 리뷰 제거 버전
reviews_en_long = reviews_en[
    reviews_en["word_count"] >= 5
].copy()

print("전체 리뷰:", reviews.shape)
print("전처리 후:", reviews_clean.shape)
print("영어 리뷰:", reviews_en.shape)
print("영어 + 5단어 이상:", reviews_en_long.shape)

reviews_en_long.head()

전체 리뷰: (236379, 21)
전처리 후: (235853, 34)
영어 리뷰: (148915, 33)
영어 + 5단어 이상: (99814, 33)


,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,...,updated_date,author_last_played_date,last_played_date,created_month,created_year,playtime_forever_hours,playtime_last_two_weeks_hours,playtime_at_review_hours,review_length,word_count
4,213162568,2185780,english,"Fun fact, if you wanted to 100% this game, at ...",1765592015,1765592015,False,0,0,0.500000,...,2025-12-13 02:13:35,2025-03-15 21:32:16,2025-03-15 21:32:16,2025-12,2025,25.650000,0.0,25.650000,149,28
24,195625586,2185780,english,"i'd give it a 3.5. Not terrible, not great either",1748180457,1748180457,False,1,0,0.474296,...,2025-05-25 13:40:57,2025-05-25 13:38:41,2025-05-25 13:38:41,2025-05,2025,0.516667,0.0,0.516667,49,10
27,193109925,2185780,english,"Really fun game. Grindier as hell, but fun.",1745076633,1745076633,True,1,0,0.516129,...,2025-04-19 15:30:33,2025-04-19 16:48:25,2025-04-19 16:48:25,2025-04,2025,5.866667,0.0,4.766667,43,8
29,191628498,2185780,english,Honestly not a bad game just kinda boring ngl,1743405777,1743405777,False,0,0,0.500000,...,2025-03-31 07:22:57,2025-03-31 07:12:44,2025-03-31 07:12:44,2025-03,2025,0.916667,0.0,0.916667,45,9
31,190296487,2185780,english,Was having fun until I got an obnoxious rocket...,1742071754,1742071778,False,3,1,0.551425,...,2025-03-15 20:49:38,2025-03-15 20:43:24,2025-03-15 20:43:24,2025-03,2025,0.933333,0.0,0.933333,353,72


In [46]:
reviews_en_long.to_csv('../../../data/processed/eng_review.csv', index=False, encoding="utf-8-sig")
